In [1]:
# -------------- setup --------------
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.signal import butter, sosfiltfilt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from mne.decoding import CSP

### More tight filter boundries 

In [2]:
FS = 250

CHANNELS = ["C3", "CZ", "C4"]

channel_sets = {
    "all_4": ["FZ", "C3", "CZ", "C4"],
    "central_3": ["C3", "CZ", "C4"],
    "motor_2": ["C3", "C4"],
}

frequency_bands = {
    "mu": (8, 12),
    "beta": (13, 30),
    "mu_beta": (8, 30),
    "wider_MI": (7, 35),
}

LOW_CUT = 8
HIGH_CUT = 30

RANDOM_STATE = 42

In [3]:
MANIFEST_PATH = Path(
    "../../data/Processed/final_clean_manifest.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)

print("Trials:", len(manifest))
print(manifest["label"].value_counts())
display(manifest.head())

Trials: 1935
label
right    971
left     964
Name: count, dtype: int64


,trial_id,file_path,label
0,1,..\..\data\Processed\final_clean_trials\cellul...,left
1,2,..\..\data\Processed\final_clean_trials\cellul...,right
2,3,..\..\data\Processed\final_clean_trials\cellul...,left
3,4,..\..\data\Processed\final_clean_trials\cellul...,left
4,5,..\..\data\Processed\final_clean_trials\cellul...,left


In [4]:
assert set(manifest["label"].unique()) == {"left", "right"}
assert manifest["trial_id"].is_unique

print("Manifest looks good.")

Manifest looks good.


In [5]:
ALL_CHANNELS = ["FZ", "C3", "CZ", "C4"]

def load_trials(manifest):
    X = []
    y = []

    for _, row in manifest.iterrows():

        df = pd.read_csv(row["file_path"])

        trial = df[ALL_CHANNELS].values.T

        X.append(trial)
        y.append(row["label"])

    return np.array(X), np.array(y)


X_all, y_labels = load_trials(manifest)

print("X shape:", X_all.shape)
print("Labels:", np.unique(y_labels, return_counts=True))

X shape: (1935, 4, 2500)
Labels: (array(['left', 'right'], dtype='<U5'), array([964, 971]))


### Encoding left into 0 and right to 1 

In [6]:
y = pd.Series(y_labels).str.lower().map({
    "left": 0,
    "right": 1
}).to_numpy()

print("Encoded labels:")
print(np.unique(y, return_counts=True))

Encoded labels:
(array([0, 1]), array([964, 971]))


### Data split 80/20

In [7]:
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train:", X_train_all.shape)
print("Test :", X_test_all.shape)

print("Train labels:", np.unique(y_train, return_counts=True))
print("Test labels :", np.unique(y_test, return_counts=True))

Train: (1548, 4, 2500)
Test : (387, 4, 2500)
Train labels: (array([0, 1]), array([771, 777]))
Test labels : (array([0, 1]), array([193, 194]))


In [8]:
channel_indices = {
    channel: idx
    for idx, channel in enumerate(ALL_CHANNELS)
}


def select_channels(X, selected_channels):

    indices = [
        channel_indices[ch]
        for ch in selected_channels
    ]

    return X[:, indices, :]

In [9]:
X_motor = select_channels(
    X_train_all,
    ["C3", "C4"]
)

print(X_motor.shape)

(1548, 2, 2500)


In [10]:
from scipy.signal import butter, sosfiltfilt


def bandpass_data(X, low, high, fs=250):

    sos = butter(
        4,
        [low, high],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    return sosfiltfilt(
        sos,
        X,
        axis=-1
    )

In [11]:
X_beta = bandpass_data(
    X_train_all,
    13,
    30
)

print(X_beta.shape)

(1548, 4, 2500)


In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
results = []

for channel_name, selected_channels in channel_sets.items():

    print("=" * 60)
    print("Channels:", channel_name, selected_channels)

    # Select channels first
    X_channels = select_channels(
        X_train_all,
        selected_channels
    )

    for band_name, (low, high) in frequency_bands.items():

        print(
            f"Testing {channel_name} | "
            f"{band_name} ({low}-{high} Hz)"
        )

        # Apply frequency filter
        X_filtered = bandpass_data(
            X_channels,
            low,
            high,
            fs=250
        )

        # CSP components cannot exceed number of channels
        n_components = min(
            4,
            len(selected_channels)
        )

        csp = CSP(
            n_components=n_components,
            reg="ledoit_wolf",
            log=True,
            norm_trace=False
        )

        svm = SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        )

        model = Pipeline([
            ("csp", csp),
            ("svm", svm)
        ])

        scores = cross_val_score(
            model,
            X_filtered,
            y_train,
            cv=cv,
            scoring="balanced_accuracy",
            error_score="raise"
        )

        results.append({
            "channels": channel_name,
            "channel_list": ",".join(selected_channels),
            "band": band_name,
            "low_hz": low,
            "high_hz": high,
            "mean_cv_accuracy": scores.mean(),
            "std_cv_accuracy": scores.std()
        })

        print(
            f"Mean CV: {scores.mean():.4f}"
        )

        print(
            f"Std: {scores.std():.4f}"
        )

        print()

Channels: all_4 ['FZ', 'C3', 'CZ', 'C4']
Testing all_4 | mu (8-12 Hz)
Computing rank from data with rank=None
    Using tolerance 1.1e+02 (2.2e-16 eps * 4 dim * 1.3e+17  max singular value)
    Estimated rank (data): 4
    data: rank 4 computed from 4 data channels with 0 projectors
Reducing data rank from 4 -> 4
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Computing rank from data with rank=None
    Using tolerance 1.1e+02 (2.2e-16 eps * 4 dim * 1.2e+17  max singular value)
    Estimated rank (data): 4
    data: rank 4 computed from 4 data channels with 0 projectors
Reducing data rank from 4 -> 4
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Computing rank from data with rank=None
    Using tolerance 1.1e+02 (2.2e-16 eps * 4 dim * 1.3e+17  max singular value)
    Estimated rank (data): 4
    data: rank 4 computed from 4 data channels with 0 projectors
Reducing d

In [14]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "mean_cv_accuracy",
    ascending=False
).reset_index(drop=True)

display(results_df)

,channels,channel_list,band,low_hz,high_hz,mean_cv_accuracy,std_cv_accuracy
0,central_3,"C3,CZ,C4",wider_MI,7,35,0.529791,0.017021
1,central_3,"C3,CZ,C4",beta,13,30,0.529092,0.019159
2,central_3,"C3,CZ,C4",mu_beta,8,30,0.525887,0.019247
3,all_4,"FZ,C3,CZ,C4",beta,13,30,0.521303,0.028124
4,all_4,"FZ,C3,CZ,C4",mu_beta,8,30,0.519472,0.022884
5,central_3,"C3,CZ,C4",mu,8,12,0.518819,0.022638
6,all_4,"FZ,C3,CZ,C4",wider_MI,7,35,0.511688,0.031199
7,all_4,"FZ,C3,CZ,C4",mu,8,12,0.503862,0.034787
8,motor_2,"C3,C4",beta,13,30,0.498089,0.041744
9,motor_2,"C3,C4",wider_MI,7,35,0.491890,0.024471


In [15]:
display_df = results_df.copy()

display_df["mean_cv_accuracy"] = (
    display_df["mean_cv_accuracy"] * 100
).round(2)

display_df["std_cv_accuracy"] = (
    display_df["std_cv_accuracy"] * 100
).round(2)

display(display_df)

,channels,channel_list,band,low_hz,high_hz,mean_cv_accuracy,std_cv_accuracy
0,central_3,"C3,CZ,C4",wider_MI,7,35,52.98,1.70
1,central_3,"C3,CZ,C4",beta,13,30,52.91,1.92
2,central_3,"C3,CZ,C4",mu_beta,8,30,52.59,1.92
3,all_4,"FZ,C3,CZ,C4",beta,13,30,52.13,2.81
4,all_4,"FZ,C3,CZ,C4",mu_beta,8,30,51.95,2.29
5,central_3,"C3,CZ,C4",mu,8,12,51.88,2.26
6,all_4,"FZ,C3,CZ,C4",wider_MI,7,35,51.17,3.12
7,all_4,"FZ,C3,CZ,C4",mu,8,12,50.39,3.48
8,motor_2,"C3,C4",beta,13,30,49.81,4.17
9,motor_2,"C3,C4",wider_MI,7,35,49.19,2.45


In [16]:
best = results_df.iloc[0]

print("Best configuration")
print("------------------")

print("Channels:", best["channel_list"])
print(
    "Frequency band:",
    f'{best["low_hz"]}-{best["high_hz"]} Hz'
)

print(
    "Mean CV accuracy:",
    f'{best["mean_cv_accuracy"] * 100:.2f}%'
)

print(
    "CV std:",
    f'{best["std_cv_accuracy"] * 100:.2f}%'
)

Best configuration
------------------
Channels: C3,CZ,C4
Frequency band: 7-35 Hz
Mean CV accuracy: 52.98%
CV std: 1.70%


### CSP Channel/Frequency Search Conclusion

- 12 channel/frequency configurations were evaluated using 5-fold cross-validation.
- The best configuration was C3, CZ and C4 with 7–35 Hz filtering.
- Best mean balanced accuracy was only **52.98% ± 1.70%**.
- CAR also reduced CSP performance.
- Therefore, standard single-band CSP + SVM does not provide strong class separation on this dataset.
- The next classical experiment will use Filter-Bank CSP (FBCSP), followed by EEGNet for learning richer temporal-spatial features.